In [1]:
!pip install transformers datasets sentencepiece evaluate

import json
import re
from tqdm import tqdm
from datasets import Dataset
from transformers import LEDTokenizer, LEDForConditionalGeneration, Trainer, TrainingArguments
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [3]:
def extract_answer(item):
    ans = item.get("answer", {})

    if isinstance(ans, dict) and "answer" in ans and ans["answer"]:
        obj = ans["answer"][0]

        if isinstance(obj, dict):
            if "label" in obj and isinstance(obj["label"], dict):
                if obj["label"].get("en"):
                    return str(obj["label"]["en"])

            if "name" in obj:
                return str(obj["name"])

        return str(obj)

    if isinstance(ans, dict) and "mention" in ans:
        return str(ans["mention"])

    return ""


def load_mintaka(path):
    with open(path) as f:
        data = json.load(f)

    questions, answers = [], []

    for item in data:
        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a     = load_mintaka("mintaka_dev.json")
test_q, test_a   = load_mintaka("mintaka_test.json")

In [4]:
def create_tuples_no_ned(questions, answers):
    inputs, labels = [], []

    for q, ans in zip(questions, answers):
        inputs.append(f"question: {q}")
        labels.append(ans)

    return inputs, labels


train_inp, train_lab = create_tuples_no_ned(train_q, train_a)
dev_inp, dev_lab     = create_tuples_no_ned(dev_q, dev_a)
test_inp, test_lab   = create_tuples_no_ned(test_q, test_a)

In [5]:
tokenizer = LEDTokenizer.from_pretrained("allenai/led-base-16384")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [6]:
def make_dataset_led(inputs, labels):
    ds = Dataset.from_dict({
        "input_text": inputs,
        "target_text": labels
    })

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            max_length=512,   # shorter since no context
            padding="max_length",
            truncation=True
        )

        labels_tok = tokenizer(
            batch["target_text"],
            max_length=32,
            padding="max_length",
            truncation=True
        )

        model_inputs["labels"] = labels_tok["input_ids"]

        # 🔥 REQUIRED for Longformer
        global_attention_mask = []
        for ids in model_inputs["input_ids"]:
            mask = [0] * len(ids)
            mask[0] = 1   # only first token global
            global_attention_mask.append(mask)

        model_inputs["global_attention_mask"] = global_attention_mask

        return model_inputs

    ds = ds.map(tokenize, batched=True)

    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "global_attention_mask", "labels"]
    )

    return ds


train_dataset = make_dataset_led(train_inp, train_lab)
dev_dataset   = make_dataset_led(dev_inp, dev_lab)
test_dataset  = make_dataset_led(test_inp, test_lab)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = LEDForConditionalGeneration.from_pretrained(
    "allenai/led-base-16384"
).to(device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/648M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie led.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie led.shared.weight to led.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./led_no_ned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=2,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.213786,0.223542


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]